## Cell 1 — Imports
Chạy 1 lần khi khởi động runtime.

In [ ]:
from google.colab import drive, userdata
from __future__ import annotations

drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/vietlink_chatbot


In [ ]:
!pip install -q sentence-transformers chromadb google-genai openai anthropic ipywidgets


In [ ]:
from __future__ import annotations

import json, os, re, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional

import chromadb
import openai
import anthropic
from google import genai
from google.genai import types as genai_types

import ipywidgets as widgets
from IPython.display import display, clear_output


## Cell 2 — Config + Validation + Provider Registry
Nguồn sự thật duy nhất cho configuration.  
**Không có hidden/default value ở bất kỳ đâu.**


In [ ]:
# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------

class ConfigurationError(Exception):
    pass

class SecretNotFoundError(Exception):
    pass


# ---------------------------------------------------------------------------
# Provider Registry — fixed models, fixed KB paths, không để user chỉnh
# ---------------------------------------------------------------------------

PROVIDER_REGISTRY: Dict[str, Dict[str, str]] = {
    "gpt": {
        "generation_model":  "gpt-5",
        "light_model":       "gpt-5-mini",      # dùng cho Prompt 0 normalization
        "kb_path":           "",                 # điền đường dẫn results/gpt/kbs.json
        "chroma_dir":        "./chroma_gpt",
        "chroma_collection": "kb_gpt",
        "secret_key":        "OPENAI_API_KEY",
    },
    "claude": {
        "generation_model":  "claude-sonnet-5",
        "light_model":       "claude-haiku-4-5-20251001",
        "kb_path":           "",                 # điền đường dẫn results/claude/kbs.json
        "chroma_dir":        "./chroma_claude",
        "chroma_collection": "kb_claude",
        "secret_key":        "ANTHROPIC_API_KEY",
    },
    "gemini": {
        "generation_model":  "gemini-flash-latest",
        "light_model":       "gemini-flash-lite-latest",
        "kb_path":           "",                 # điền đường dẫn results/gemini/kbs.json
        "chroma_dir":        "./chroma_gemini",
        "chroma_collection": "kb_gemini",
        "secret_key":        "GOOGLE_API_KEY",
    },
}

FIXED_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# Tự động điền kb_path bằng cách scan Drive —
# kết quả ghi vào PROVIDER_REGISTRY, không tạo biến global rải rác.
_filename = "kbs.json"
for _root, _dirs, _files in os.walk("/content/drive"):
    if _filename not in _files:
        continue
    _parent = os.path.basename(os.path.dirname(_root))
    _provider = os.path.basename(_root)
    if _parent == "results" and _provider in PROVIDER_REGISTRY:
        PROVIDER_REGISTRY[_provider]["kb_path"] = os.path.join(_root, _filename)

for _p, _cfg in PROVIDER_REGISTRY.items():
    _status = "✅" if _cfg["kb_path"] else "❌ NOT FOUND"
    print(f"  {_p}: {_status}  {_cfg['kb_path']}")


In [ ]:
# ---------------------------------------------------------------------------
# RAGConfig — nguồn sự thật duy nhất cho configuration runtime.
# Mọi field mặc định None; applied = False.
# KHÔNG field nào có default value được dùng ngầm.
# ---------------------------------------------------------------------------

@dataclass
class RAGConfig:
    provider:             Optional[str]   = None
    temperature:          Optional[float] = None
    max_tokens:           Optional[int]   = None
    top_k:                Optional[int]   = None
    similarity_threshold: Optional[float] = None
    applied:              bool            = False   # chỉ True sau khi Apply thành công

    def validate(self) -> None:
        """Raise ConfigurationError nêu rõ đúng field bị thiếu/sai.
        Không có nhánh nào âm thầm điền giá trị thay thế."""
        if self.provider is None:
            raise ConfigurationError("Provider has not been configured.")
        if self.provider not in PROVIDER_REGISTRY:
            raise ConfigurationError(f"Unsupported provider: {self.provider!r}.")
        if not PROVIDER_REGISTRY[self.provider]["kb_path"]:
            raise ConfigurationError(
                f"KB file not found for provider '{self.provider}'. "
                "Run kb_mining for this provider first."
            )
        if self.temperature is None:
            raise ConfigurationError("Temperature has not been configured.")
        if not (0.0 <= self.temperature <= 2.0):
            raise ConfigurationError("Temperature must be between 0.0 and 2.0.")
        if self.max_tokens is None:
            raise ConfigurationError("Max tokens has not been configured.")
        if self.max_tokens <= 0:
            raise ConfigurationError("Max tokens must be a positive integer.")
        if self.top_k is None:
            raise ConfigurationError("Top K has not been configured.")
        if self.top_k <= 0:
            raise ConfigurationError("Top K must be a positive integer.")
        if self.similarity_threshold is None:
            raise ConfigurationError("Similarity threshold has not been configured.")
        if not (0.0 <= self.similarity_threshold <= 1.0):
            raise ConfigurationError("Similarity threshold must be between 0.0 and 1.0.")

    def is_ready(self) -> bool:
        """True khi configuration đã được Apply thành công (applied = True)
        VÀ validate() không raise lỗi."""
        if not self.applied:
            return False
        try:
            self.validate()
            return True
        except ConfigurationError:
            return False

    def summary(self) -> str:
        reg = PROVIDER_REGISTRY.get(self.provider or "", {})
        return (
            f"Provider:             {self.provider}\n"
            f"Generation model:     {reg.get('generation_model', 'N/A')}\n"
            f"Embedding model:      {FIXED_EMBEDDING_MODEL}\n"
            f"Temperature:          {self.temperature}\n"
            f"Max tokens:           {self.max_tokens}\n"
            f"Top K:                {self.top_k}\n"
            f"Similarity threshold: {self.similarity_threshold}\n"
            f"Applied:              {self.applied}"
        )


## Cell 3 — ClientManager + FixedEmbeddingManager
- **ClientManager**: lazy-init + cache, 1 client/provider, không tạo trùng.
- **FixedEmbeddingManager**: khởi tạo 1 lần, dùng chung cho cả 3 provider,
  không recreate khi đổi provider hay Apply Configuration.


In [ ]:
# ---------------------------------------------------------------------------
# ClientManager
# ---------------------------------------------------------------------------

class ClientManager:
    """Lazy-init và cache client theo provider.
    Gọi get() nhiều lần với cùng provider → cùng 1 instance."""

    def __init__(self) -> None:
        self._cache: Dict[str, Any] = {}

    def _load_secret(self, provider: str) -> str:
        secret_name = PROVIDER_REGISTRY[provider]["secret_key"]
        try:
            key = userdata.get(secret_name)
        except Exception as e:
            raise SecretNotFoundError(
                f"{secret_name} is not configured in Colab Secrets."
            ) from e
        if not key:
            raise SecretNotFoundError(
                f"{secret_name} is not configured in Colab Secrets."
            )
        return key

    def get(self, provider: str) -> Any:
        if provider in self._cache:
            return self._cache[provider]

        key = self._load_secret(provider)
        if provider == "gpt":
            client = openai.OpenAI(api_key=key)
        elif provider == "claude":
            client = anthropic.Anthropic(api_key=key)
        elif provider == "gemini":
            client = genai.Client(api_key=key)
        else:
            raise ConfigurationError(f"Unsupported provider: {provider!r}.")

        self._cache[provider] = client
        return client


In [ ]:
# ---------------------------------------------------------------------------
# FixedEmbeddingManager
# ---------------------------------------------------------------------------

class FixedEmbeddingManager:
    """Khởi tạo embedding model đúng 1 lần; không recreate khi đổi provider."""

    def __init__(self, model_name: str) -> None:
        from sentence_transformers import SentenceTransformer
        self.model_name = model_name
        self._model = SentenceTransformer(model_name)
        print(f"Embedding model loaded: {model_name}")

    def embed(self, text: str) -> List[float]:
        return self._model.encode(text, normalize_embeddings=True).tolist()

    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        return self._model.encode(texts, normalize_embeddings=True).tolist()


## Cell 4 — KB + VectorDBManager + index_all_validated
- `VectorDBManager` quản lý 3 ChromaDB riêng biệt theo provider.
- `index_all_validated()` giữ nguyên chức năng cũ, nhận tham số tường minh.
- Đổi provider → `VectorDBManager.switch_provider()` re-embed và rebuild
  ChromaDB của provider mới (luôn rebuild theo quyết định thiết kế đã chốt).


In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_kb_data(path: str) -> Dict[str, Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        entries: List[Dict[str, Any]] = json.load(f)
    return {e["knowledge_id"]: e for e in entries}


def get_many(kb_data: Dict[str, Dict[str, Any]], ids: List[str]) -> List[Dict[str, Any]]:
    return [kb_data[i] for i in ids if i in kb_data]


def index_all_validated(
    kb_data: Dict[str, Dict[str, Any]],
    embedding_manager: FixedEmbeddingManager,
    collection,          # chromadb collection instance
) -> None:
    """Giữ nguyên chức năng cũ; nhận tham số tường minh thay vì đọc global.
    Embed tất cả entry (không lọc theo status — codebase cũ làm vậy)."""
    entries = list(kb_data.values())
    if not entries:
        print("  index_all_validated: kb_data rỗng, không có gì để index.")
        return
    triggers = [e["trigger"] for e in entries]
    vectors = embedding_manager.embed_batch(triggers)
    for entry, vector in zip(entries, vectors):
        collection.upsert(
            ids=[entry["knowledge_id"]],
            embeddings=[vector],
            metadatas=[{"status": entry.get("status", "Validated")}],
        )
    print(f"  index_all_validated: indexed {len(entries)} entries.")


In [ ]:
# ---------------------------------------------------------------------------
# VectorDBManager
# ---------------------------------------------------------------------------

@dataclass
class SearchResult:
    id: str
    score: float


class VectorDBManager:
    """Quản lý 3 ChromaDB riêng cho 3 provider.
    - switch_provider(): luôn rebuild ChromaDB của provider mới (chọn thiết kế đơn giản,
      tránh edge case KB cũ/mới không khớp).
    - rebuild(): action riêng, chỉ gọi khi user bấm 'Rebuild Current VectorDB'.
    - Không tự động rebuild khi chỉ đổi top_k/temperature/max_tokens.
    """

    def __init__(self, embedding_manager: FixedEmbeddingManager) -> None:
        self._embedding_manager = embedding_manager
        self._chroma_clients: Dict[str, Any]        = {}   # provider -> PersistentClient
        self._collections:    Dict[str, Any]        = {}   # provider -> collection
        self._kb_data:        Dict[str, Dict[str, Dict[str, Any]]] = {}  # provider -> kb_data
        self._active_provider: Optional[str]        = None

    # ------------------------------------------------------------------
    def _get_or_create_chroma(self, provider: str):
        if provider in self._chroma_clients:
            return self._chroma_clients[provider], self._collections[provider]
        cfg = PROVIDER_REGISTRY[provider]
        persist_dir = os.path.abspath(cfg["chroma_dir"])
        os.makedirs(persist_dir, exist_ok=True)
        client = chromadb.PersistentClient(path=persist_dir)
        collection = client.get_or_create_collection(
            name=cfg["chroma_collection"],
            metadata={"hnsw:space": "cosine"},
        )
        self._chroma_clients[provider] = client
        self._collections[provider]    = collection
        return client, collection

    def _reset_collection(self, provider: str) -> None:
        """Xoá và tạo lại collection cho provider (chỉ dữ liệu, không xoá thư mục)."""
        client, _ = self._get_or_create_chroma(provider)
        cfg = PROVIDER_REGISTRY[provider]
        client.delete_collection(cfg["chroma_collection"])
        collection = client.get_or_create_collection(
            name=cfg["chroma_collection"],
            metadata={"hnsw:space": "cosine"},
        )
        self._collections[provider] = collection

    # ------------------------------------------------------------------
    def switch_provider(self, provider: str) -> None:
        """Đổi sang provider mới:
        1. Load Raw KB của provider đó.
        2. Reset (xoá) ChromaDB cũ của provider đó.
        3. Re-embed bằng SAME fixed embedding model.
        4. Rebuild ChromaDB của provider đó.
        """
        if provider not in PROVIDER_REGISTRY:
            raise ConfigurationError(f"Unsupported provider: {provider!r}.")
        cfg = PROVIDER_REGISTRY[provider]
        if not cfg["kb_path"]:
            raise ConfigurationError(
                f"KB file not found for provider '{provider}'. "
                "Run kb_mining for this provider first."
            )

        print(f"[VectorDBManager] Switching to provider '{provider}'...")

        # 1. Load KB
        kb = load_kb_data(cfg["kb_path"])
        self._kb_data[provider] = kb
        print(f"  Loaded {len(kb)} KB entries from {cfg['kb_path']}")

        # 2. Đảm bảo chroma client/collection tồn tại, rồi reset
        self._get_or_create_chroma(provider)
        self._reset_collection(provider)
        print(f"  ChromaDB reset: {cfg['chroma_dir']}")

        # 3 + 4. Re-embed và index
        collection = self._collections[provider]
        index_all_validated(kb, self._embedding_manager, collection)

        self._active_provider = provider
        print(f"[VectorDBManager] Provider '{provider}' ready.")

    # ------------------------------------------------------------------
    def rebuild(self, provider: str) -> None:
        """Rebuild ChromaDB của provider hiện tại — chỉ được gọi từ nút
        'Rebuild Current VectorDB' do user chủ động bấm."""
        if provider not in self._kb_data:
            # KB chưa được load lần nào → switch_provider trước
            self.switch_provider(provider)
            return
        print(f"[VectorDBManager] Rebuilding ChromaDB for '{provider}'...")
        self._reset_collection(provider)
        collection = self._collections[provider]
        index_all_validated(self._kb_data[provider], self._embedding_manager, collection)
        print(f"[VectorDBManager] Rebuild complete for '{provider}'.")

    # ------------------------------------------------------------------
    def search(
        self,
        provider: str,
        vector: List[float],
        top_k: int,
        similarity_threshold: float,
    ) -> List[SearchResult]:
        """Search trong ChromaDB của đúng provider."""
        if provider not in self._collections:
            raise ConfigurationError(
                f"VectorDB for provider '{provider}' has not been initialized. "
                "Apply configuration first."
            )
        collection = self._collections[provider]
        raw = collection.query(query_embeddings=[vector], n_results=top_k)
        ids       = raw.get("ids",       [[]])[0]
        distances = raw.get("distances", [[]])[0]
        results   = [SearchResult(id=i, score=1.0 - d) for i, d in zip(ids, distances)]
        return [r for r in results if r.score >= similarity_threshold]

    # ------------------------------------------------------------------
    def get_kb_data(self, provider: str) -> Dict[str, Dict[str, Any]]:
        if provider not in self._kb_data:
            raise ConfigurationError(
                f"KB for provider '{provider}' has not been loaded. "
                "Apply configuration first."
            )
        return self._kb_data[provider]

    @property
    def active_provider(self) -> Optional[str]:
        return self._active_provider


## Cell 5 — RAGSystem
`RAGSystem` khởi tạo 1 lần, dùng lại mãi.  
UI chỉ gọi `rag_system.apply_configuration()` và `rag_system.run_query()`.


In [ ]:
# ---------------------------------------------------------------------------
# Prompts (giữ nguyên từ codebase cũ)
# ---------------------------------------------------------------------------

SHORT_INPUT_TOKEN_THRESHOLD = 40
LONG_INPUT_TOKEN_THRESHOLD  = 150
MAX_CLARIFICATION_ROUNDS    = 3

_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?…])\s+|\n+")

def token_count(text: str) -> int:
    return len(text.split())

def split_into_sentences(text: str) -> List[str]:
    parts = _SENTENCE_SPLIT_RE.split(text.strip())
    return [p.strip() for p in parts if p.strip()]


PROMPT_0_TEMPLATE = """\
You are given a user request, which may be long and contain several \
kinds of information mixed together: background context, constraints \
already stated, and the actual request. Your task is to split it into \
3 parts, WITHOUT paraphrasing or adding anything.

1. core_request: EXACTLY what the user is asking to be done.
2. stated_info: every concrete detail already provided (one line per item).
3. background_context: everything else.

RULES: Do not infer. Empty string/array if nothing fits. Preserve language.

Output JSON only.

Schema:
{{
  "core_request": "string",
  "stated_info": ["string"],
  "background_context": "string"
}}

<raw_request>
{raw_request}
</raw_request>
"""


PROMPT_3_TEMPLATE = """\
You are given a NEW user request and a list of knowledge base entries.
Each entry describes one type of ambiguity that has occurred in the past,
with its trigger, detection_rules, and a sample clarifying question.

You are inside an ongoing conversation — earlier turns are the source of
truth for what has been resolved.

Task: ask EXACTLY 1 clarifying question if something is still missing,
or rewrite the request once everything is clear / round limit is reached.

RULES:
1. matched=true only when detection_rules genuinely match — not just trigger.
2. resolved=true if info is present in the request OR earlier conversation.
3. If unresolved AND round_number < {max_rounds}: status="ask", 1 question,
   prioritize highest-priority entry.
4. next_question natural, no internal KB terminology.
5. HARD LIMIT at round {max_rounds}: status="done", make assumptions,
   record in assumptions_made.
6. When all resolved / nothing matched: status="done".
7. clarified_request = minimal edit (INSERT/REPLACE/APPEND only).

Output JSON only.

Schema:
{{
  "matches": [{{"knowledge_id":"string","matched":boolean,
                "resolved":boolean,"match_reasoning":"string"}}],
  "status": "ask" | "done",
  "next_question": "string",
  "clarified_request": "string",
  "assumptions_made": ["string"]
}}

<new_request>{new_request}</new_request>
<candidate_kb_entries>{candidate_kb_entries}</candidate_kb_entries>
<round_number>{round_number}</round_number>
"""


In [ ]:
# ---------------------------------------------------------------------------
# LLM call helpers
# ---------------------------------------------------------------------------

def _parse_json_output(raw: str) -> Dict[str, Any]:
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", cleaned, re.DOTALL)
        return json.loads(m.group(0)) if m else {}


@dataclass
class NormalizedRequest:
    core_request:       str
    stated_info:        List[str] = field(default_factory=list)
    background_context: str       = ""


def _llm_call_oneshot(
    provider: str,
    client:   Any,
    model:    str,
    prompt:   str,
    temperature: float,
    max_tokens:  int,
    json_mode:   bool = False,
) -> str:
    """Single-turn LLM call cho cả 3 provider."""
    if provider == "gemini":
        cfg = genai_types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
            response_mime_type="application/json" if json_mode else None,
        )
        return client.models.generate_content(model=model, contents=prompt, config=cfg).text

    if provider == "gpt":
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_completion_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
            )
        except openai.BadRequestError:
            # Reasoning-tier models không nhận temperature — bỏ qua silently
            # (quyết định thiết kế đã chốt: không báo lỗi, không fallback giá trị khác)
            resp = client.chat.completions.create(
                model=model,
                max_completion_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
            )
        return resp.choices[0].message.content

    if provider == "claude":
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")

    raise ConfigurationError(f"Unsupported provider: {provider!r}.")


def _llm_chat_send(
    provider: str,
    client:   Any,
    model:    str,
    session,
    prompt:   str,
    temperature: float,
    max_tokens:  int,
) -> str:
    """Multi-turn: Gemini dùng chat session SDK; GPT/Claude dùng messages list."""
    if provider == "gemini":
        cfg = genai_types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
        )
        return session.send_message(prompt, config=cfg).text

    session.append({"role": "user", "content": prompt})
    text = _llm_call_oneshot(provider, client, model, prompt, temperature, max_tokens)
    session.append({"role": "assistant", "content": text})
    return text


def _llm_chat_answer(provider: str, session, answer: str) -> None:
    """Đưa câu trả lời của user vào lịch sử session."""
    if provider == "gemini":
        session.send_message(answer)
    else:
        session.append({"role": "user", "content": answer})


In [ ]:
# ---------------------------------------------------------------------------
# Retriever
# ---------------------------------------------------------------------------

class Retriever:
    """Đọc top_k/similarity_threshold trực tiếp từ config tại thời điểm gọi.
    Không cache tham số retrieval — đảm bảo luôn dùng giá trị mới nhất từ RAGConfig."""

    def __init__(
        self,
        embedding_manager: FixedEmbeddingManager,
        vector_db_manager: VectorDBManager,
        client_manager:    ClientManager,
    ) -> None:
        self._em  = embedding_manager
        self._vdb = vector_db_manager
        self._cm  = client_manager

    def _call_prompt_0(self, provider: str, raw_request: str) -> NormalizedRequest:
        cfg    = PROVIDER_REGISTRY[provider]
        client = self._cm.get(provider)
        model  = cfg["light_model"]
        prompt = PROMPT_0_TEMPLATE.format(raw_request=raw_request)
        raw    = _llm_call_oneshot(provider, client, model, prompt,
                                   temperature=0, max_tokens=512, json_mode=True)
        data   = _parse_json_output(raw)
        return NormalizedRequest(
            core_request=data.get("core_request", "") or "",
            stated_info=data.get("stated_info", []) or [],
            background_context=data.get("background_context", "") or "",
        )

    @dataclass
    class RetrievalResult:
        candidates:     List[Dict[str, Any]]
        query_strategy: str
        normalized:     Optional[NormalizedRequest] = None

    def retrieve(
        self,
        config:  "RAGConfig",
        query:   str,
    ) -> "Retriever.RetrievalResult":
        """config phải đã passed validate() trước khi gọi đây."""
        provider    = config.provider
        top_k       = config.top_k
        sim_thresh  = config.similarity_threshold
        kb_data     = self._vdb.get_kb_data(provider)
        n_tokens    = token_count(query)

        if n_tokens <= SHORT_INPUT_TOKEN_THRESHOLD:
            vec        = self._em.embed(query)
            candidates = self._vdb.search(provider, vec, top_k, sim_thresh)
            entries    = get_many(kb_data, [r.id for r in candidates])
            return self.RetrievalResult(entries, "direct")

        if n_tokens <= LONG_INPUT_TOKEN_THRESHOLD:
            sentences = split_into_sentences(query)
            best: Dict[str, float] = {}
            for vec in self._em.embed_batch(sentences):
                for r in self._vdb.search(provider, vec, top_k, sim_thresh):
                    if r.id not in best or r.score > best[r.id]:
                        best[r.id] = r.score
            sorted_ids = sorted(best, key=lambda k: -best[k])[:top_k]
            entries    = get_many(kb_data, sorted_ids)
            return self.RetrievalResult(entries, "sentence_chunk")

        normalized = self._call_prompt_0(provider, query)
        query_text = normalized.core_request or query
        vec        = self._em.embed(query_text)
        candidates = self._vdb.search(provider, vec, top_k, sim_thresh)
        entries    = get_many(kb_data, [r.id for r in candidates])
        return self.RetrievalResult(entries, "normalized", normalized)


In [ ]:
# ---------------------------------------------------------------------------
# Generator
# ---------------------------------------------------------------------------

_ANSWER_PROMPT = """\
You are a helpful assistant. Use the knowledge base context below to
answer the user's query. If the context is empty or irrelevant, answer
from your own knowledge and say so explicitly.

<context>
{context}
</context>

<query>
{query}
</query>
"""

def _build_context(entries: List[Dict[str, Any]]) -> str:
    if not entries:
        return "(no relevant knowledge base entries found)"
    blocks = []
    for e in entries:
        blocks.append(
            f"[{e.get('knowledge_id','')}] {e.get('title','')}\n"
            f"  Category:       {e.get('category','')}\n"
            f"  Description:    {e.get('description','')}\n"
            f"  Recommendation: {e.get('recommendation','')}"
        )
    return "\n\n".join(blocks)


class Generator:
    """Đọc model/temperature/max_tokens trực tiếp từ RAGConfig tại thời điểm gọi.
    Không có fallback/default value nào trong class này."""

    def __init__(self, client_manager: ClientManager) -> None:
        self._cm = client_manager

    def generate(self, query: str, context_entries: List[Dict[str, Any]], config: "RAGConfig") -> str:
        config.validate()
        provider = config.provider
        client   = self._cm.get(provider)
        model    = PROVIDER_REGISTRY[provider]["generation_model"]
        prompt   = _ANSWER_PROMPT.format(
            context=_build_context(context_entries), query=query
        )
        return _llm_call_oneshot(
            provider, client, model, prompt,
            temperature=config.temperature, max_tokens=config.max_tokens,
        )


In [ ]:
# ---------------------------------------------------------------------------
# RAGSystem — orchestrator, khởi tạo 1 lần
# ---------------------------------------------------------------------------

class RAGSystem:
    def __init__(self) -> None:
        self.config      = RAGConfig()         # mọi field None, applied=False
        self._cm         = ClientManager()
        self._em         = FixedEmbeddingManager(FIXED_EMBEDDING_MODEL)
        self._vdb        = VectorDBManager(self._em)
        self._retriever  = Retriever(self._em, self._vdb, self._cm)
        self._generator  = Generator(self._cm)
        print("RAGSystem initialized. config.is_ready() =", self.config.is_ready())

    # ------------------------------------------------------------------
    def apply_configuration(self, candidate: RAGConfig) -> None:
        """Validate → thử lấy API client (bắt lỗi SecretNotFoundError sớm) →
        nếu provider đổi: switch_provider() (reload KB + re-embed + rebuild VectorDB) →
        cập nhật self.config. Nếu bất kỳ bước nào lỗi, self.config giữ nguyên cũ."""
        candidate.validate()
        self._cm.get(candidate.provider)  # kiểm tra API key trước khi commit

        if candidate.provider != self.config.provider:
            self._vdb.switch_provider(candidate.provider)

        candidate.applied = True
        self.config       = candidate

    # ------------------------------------------------------------------
    def run_query(self, query: str) -> Dict[str, Any]:
        if not self.config.is_ready():
            raise ConfigurationError(
                "Please apply a valid configuration before running the query."
            )
        retrieval = self._retriever.retrieve(self.config, query)
        answer    = self._generator.generate(query, retrieval.candidates, self.config)
        return {
            "answer":          answer,
            "query_strategy":  retrieval.query_strategy,
            "matched_kb_ids":  [e["knowledge_id"] for e in retrieval.candidates],
        }

    # ------------------------------------------------------------------
    def rebuild_current_vdb(self) -> None:
        """Action riêng, chỉ gọi từ nút 'Rebuild Current VectorDB' —
        không bao giờ tự động gọi ở bất kỳ đâu khác trong hệ thống."""
        if not self.config.provider:
            raise ConfigurationError("No provider configured. Apply configuration first.")
        self._vdb.rebuild(self.config.provider)


# Khởi tạo 1 lần duy nhất — UI sẽ gọi rag_system.apply_configuration / run_query
rag_system = RAGSystem()


## Cell 6 — UI
Sau khi cell này chạy, mọi thao tác đều qua widget — không cần chạy lại cell nào.


In [ ]:
import builtins  # đảm bảo tham chiếu input() gốc không bị shadow

# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------

_UNSET = None

provider_dd = widgets.Dropdown(
    options=[("— chưa chọn —", _UNSET), ("GPT", "gpt"), ("Claude", "claude"), ("Gemini", "gemini")],
    value=_UNSET,
    description="Provider:",
    style={"description_width": "initial"},
)

gen_model_lbl = widgets.Label(value="Generation model: (chọn provider trước)")
emb_model_lbl = widgets.Label(value=f"Embedding model:  {FIXED_EMBEDDING_MODEL}")

temperature_sl = widgets.FloatSlider(
    min=0.0, max=2.0, step=0.05,
    description="Temperature:", style={"description_width": "initial"},
)
max_tokens_sl = widgets.IntSlider(
    min=1, max=8192, step=1,
    description="Max Tokens:", style={"description_width": "initial"},
)
top_k_sl = widgets.IntSlider(
    min=3, max=20, step=1,
    description="Top K:", style={"description_width": "initial"},
)
sim_thresh_sl = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.01,
    description="Similarity Threshold:", style={"description_width": "initial"},
)


def _on_provider_change(change: Dict[str, Any]) -> None:
    p = change["new"]
    if p is None:
        gen_model_lbl.value = "Generation model: (chọn provider trước)"
    else:
        gen_model_lbl.value = f"Generation model: {PROVIDER_REGISTRY[p]['generation_model']}  [read-only]"


provider_dd.observe(_on_provider_change, names="value")

apply_btn   = widgets.Button(description="Apply Configuration",       button_style="primary")
reset_btn   = widgets.Button(description="Reset Configuration",       button_style="warning")
rebuild_btn = widgets.Button(description="Rebuild Current VectorDB",  button_style="danger")
config_out  = widgets.Output()

query_ta  = widgets.Textarea(
    placeholder="Enter your query...",
    description="Query:",
    layout=widgets.Layout(width="100%", height="90px"),
    style={"description_width": "initial"},
)
run_btn    = widgets.Button(description="Run RAG", button_style="success")
answer_out = widgets.Output()


# ---------------------------------------------------------------------------
# Callbacks — tất cả lỗi được bắt và hiển thị trong UI, không crash notebook
# ---------------------------------------------------------------------------

def _on_apply(b) -> None:
    with config_out:
        clear_output()
        # Đọc giá trị từ widget — KHÔNG có default value nào ở đây.
        # provider và các slider không có "trạng thái trống" tự nhiên (slider
        # luôn trả một số) → validation trong RAGConfig.validate() là nơi
        # kiểm tra đúng/sai, không phải ở đây.
        candidate = RAGConfig(
            provider=provider_dd.value,
            temperature=temperature_sl.value,
            max_tokens=max_tokens_sl.value,
            top_k=top_k_sl.value,
            similarity_threshold=sim_thresh_sl.value,
        )
        try:
            rag_system.apply_configuration(candidate)
        except ConfigurationError as e:
            print(f"❌ Configuration Error\n\n{e}")
            return
        except SecretNotFoundError as e:
            print(f"❌ Secret Not Found\n\n{e}")
            return
        except Exception as e:
            print(f"❌ Error\n\n{type(e).__name__}: {e}")
            return
        print("✅ Configuration applied successfully\n")
        print(rag_system.config.summary())


def _on_reset(b) -> None:
    provider_dd.value   = _UNSET
    temperature_sl.value = temperature_sl.min
    max_tokens_sl.value  = max_tokens_sl.max
    top_k_sl.value       = top_k_sl.min
    sim_thresh_sl.value  = sim_thresh_sl.min
    rag_system.config    = RAGConfig()   # mọi field None, applied=False
    with config_out:
        clear_output()
        print("Configuration reset. Apply a new configuration before running a query.")


def _on_rebuild(b) -> None:
    with config_out:
        clear_output()
        try:
            rag_system.rebuild_current_vdb()
            print(f"✅ VectorDB rebuilt for provider '{rag_system.config.provider}'.")
        except ConfigurationError as e:
            print(f"❌ Configuration Error\n\n{e}")
        except Exception as e:
            print(f"❌ ChromaDB Error\n\n{type(e).__name__}: {e}")


def _on_run(b) -> None:
    with answer_out:
        clear_output()
        query = (query_ta.value or "").strip()
        if not query:
            print("Please enter a query.")
            return
        if not rag_system.config.is_ready():
            print("Please apply a valid configuration before running the query.")
            return
        try:
            result = rag_system.run_query(query)
        except ConfigurationError as e:
            print(f"❌ Configuration Error\n\n{e}")
            return
        except SecretNotFoundError as e:
            print(f"❌ Secret Not Found\n\n{e}")
            return
        except Exception as e:
            print(f"❌ Error\n\n{type(e).__name__}: {e}")
            return
        print(f"[strategy: {result['query_strategy']}]  "
              f"[matched: {result['matched_kb_ids']}]\n")
        print(result["answer"])


apply_btn.on_click(_on_apply)
reset_btn.on_click(_on_reset)
rebuild_btn.on_click(_on_rebuild)
run_btn.on_click(_on_run)


# ---------------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------------

ui = widgets.VBox([
    widgets.HTML("<h3>RAG Configuration</h3>"),
    provider_dd,
    gen_model_lbl,
    emb_model_lbl,
    widgets.HTML("<h4>Generation Parameters</h4>"),
    temperature_sl,
    max_tokens_sl,
    widgets.HTML("<h4>Retrieval Parameters</h4>"),
    top_k_sl,
    sim_thresh_sl,
    widgets.HBox([apply_btn, reset_btn]),
    widgets.HTML("<h4>Configuration Status</h4>"),
    config_out,
    widgets.HTML("<hr><h3>Query</h3>"),
    query_ta,
    run_btn,
    widgets.HTML("<h4>Answer</h4>"),
    answer_out,
    widgets.HTML("<hr>"),
    rebuild_btn,
])

display(ui)
